## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 11 — Self-healing from trends, not snapshots

The matrix analog of notebook 09. Where 09 read **one row** per worker —
its latest health report — and decided from that snapshot, this reads a
**window of reports** per worker and decides from the trend.

That difference is the whole point. A worker sitting at 80% CPU and
falling is healthy; a worker at 60% and climbing fast is the one about to
fail. A single row cannot tell those apart, and 09 was explicitly fixed in
an earlier sprint to take `ORDER BY timestamp DESC LIMIT 1` precisely
because mixing a stale row into a snapshot misled it.

So this notebook asks for a `(rows, columns)` slice per worker and expects
the head to reason over the shape, not a scalar. The data is built so the
snapshot answer and the trend answer **disagree** — a worker whose current
CPU looks worst is not the one in trouble.


## Parameters

In [2]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
fleet managed : Gustavo


## Generate a window of health reports per worker

Ten reports each, one minute apart. Two workers are made to look alarming
in different ways, and only one of them actually is:

- **worker2** sits high but *flat* — consistently busy, not degrading.
- **worker3** starts comfortable and climbs steeply — it is the one that
  will fall over, and its latest single reading is still lower than
  worker2's.

Each worker's config is written to its real `~/.scarlet/config.yaml`.
There is no `CSV_PATH`, and the paths carry no worker index.


In [3]:
import numpy as np
import pandas as pd
import yaml

DATA_DIR = os.environ.get("DATA_DIR", "/data")
REPORTS = 10

# Deliberately irregular paths and names: nothing may pattern-match a
# worker index out of a filename or a source name.
LAYOUT = {
    "worker1": ("sites/north/node_health.csv",        "north_node_health"),
    "worker2": ("telemetry/busy_box.csv",             "busy_box_telemetry"),
    "worker3": ("legacy/2026/unit_c/diagnostics.csv", "unit_c_diagnostics"),
    "worker4": ("inbox/health_dump.csv",              "health_dump_inbox"),
}

# (start_cpu, per-report drift). The trap is in here:
#   worker2 is high but flat      -> worst *snapshot*, fine on trend
#   worker3 starts low and climbs -> best-looking early, actually failing
PROFILE = {
    "worker1": (30.0,  0.2),
    "worker2": (82.0,  0.1),
    "worker3": (41.0,  4.6),
    "worker4": (35.0, -0.4),
}

for i, (node, (relpath, source_name)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=7000 + i)
    base, drift = PROFILE[node]
    start = pd.Timestamp("2026-01-01T00:00:00")
    cpu = np.clip(base + drift * np.arange(REPORTS) + rng.normal(0, 1.0, REPORTS), 0, 100)
    df = pd.DataFrame({
        "timestamp":      [start + pd.Timedelta(minutes=k) for k in range(REPORTS)],
        "cpu_pct":        np.round(cpu, 2),
        "mem_pct":        np.round(np.clip(cpu * 0.7 + rng.normal(0, 2.0, REPORTS), 0, 100), 2),
        "queue_depth":    np.clip((cpu / 10 + rng.normal(0, 0.8, REPORTS)).astype(int), 0, None),
        "active_tasks":   np.clip((cpu / 12 + rng.normal(0, 0.6, REPORTS)).astype(int), 0, None),
    })

    path = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [{
            "name": source_name,
            "type": "csv",
            "mode": "local",
            "description": (
                f"{REPORTS} consecutive health reports for this node, one per "
                f"minute, oldest first. Columns: timestamp, cpu_pct, mem_pct, "
                f"queue_depth, active_tasks."
            ),
            "path": path,
        }]}, fh, sort_keys=False)

    print(f"{source_name:22s} cpu {cpu[0]:5.1f} -> {cpu[-1]:5.1f}  ({drift:+.1f}/report)  {path}")

print("\nsnapshot answer (worst final cpu):",
      max(LAYOUT, key=lambda n: PROFILE[n][0] + PROFILE[n][1] * (REPORTS - 1)))
print("trend answer    (steepest climb) :",
      max(LAYOUT, key=lambda n: PROFILE[n][1]))


north_node_health      cpu  30.3 ->  31.3  (+0.2/report)  /data/sites/north/node_health.csv
busy_box_telemetry     cpu  82.3 ->  83.9  (+0.1/report)  /data/telemetry/busy_box.csv
unit_c_diagnostics     cpu  41.3 ->  81.6  (+4.6/report)  /data/legacy/2026/unit_c/diagnostics.csv
health_dump_inbox      cpu  33.7 ->  31.5  (-0.4/report)  /data/inbox/health_dump.csv

snapshot answer (worst final cpu): worker2
trend answer    (steepest climb) : worker3


## No separate failure injection

09 patched one worker's CSV after the fact to simulate a failure. Here the
degradation is already in the generated window above — worker3's climb
*is* the failure, and it is only visible across rows.


In [4]:
print("worker3's degradation is already in its generated window - nothing to inject.")


worker3's degradation is already in its generated window - nothing to inject.


## Bring the fleet up

In [5]:
AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

Gustavo owns this fleet - not launching anything.
If the wait below times out, check the Gustavo deployment.


## Wait for the fleet to come online

In [6]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## Ask the head to find the worker in trouble

The head is told nothing about which source to read, which columns exist,
or how many rows there are. It must get a window per worker and compare
first reports against last.

The snapshot answer and the trend answer disagree by construction, so an
answer naming the *currently* highest-CPU worker is the wrong one.


In [7]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# list_sources first. The head genuinely does not know any source name,
# and list_tags cannot help: it *requires* a source_name, which is the
# thing being discovered. An earlier run of this notebook failed exactly
# there - the head guessed "node_health", "system_health", "worker_01"
# and so on, found nothing, and gave up. list_sources takes no arguments
# for that reason.
#
# "all ten rows" is deliberate, and is the opposite of 09's instruction.
# 09 was fixed to take LIMIT 1 because a stale row corrupted its snapshot;
# here the rows ARE the signal, and one row cannot express a trend.
answer = await ask_scarlet_agent(
    "Find the worker that is degrading, across a fleet whose nodes each "
    "hold a window of health reports. "
    "Step 1: call list_sources, which takes no arguments, to discover "
    "what data sources exist across the fleet. You have not been told any "
    "source names and must not guess them. "
    "Step 2: for each source, use query_feature to fetch all ten rows of "
    "timestamp and cpu_pct, oldest first. "
    "Step 3: for each worker compare its FIRST report against its LAST to "
    "get the direction and size of the change. "
    "Report a small table: source name, first cpu_pct, last cpu_pct, the "
    "change, and whether it is rising or flat or falling. "
    "Then name the single worker most at risk. Be careful: the worker with "
    "the highest current cpu_pct is not necessarily the one degrading - "
    "judge by the trend across the window, not by the latest value."
)
print(answer)


Here is the analysis of each worker's health trend, based on the change from the first to the last report in their window:

| Source Name             | First CPU% | Last CPU% | Change | Trend   |
|-------------------------|------------|-----------|--------|---------|
| north_node_health       | 30.33      | 31.33     | +1.00  | Rising  |
| busy_box_telemetry      | 82.32      | 83.91     | +1.59  | Rising  |
| unit_c_diagnostics      | 41.28      | 81.61     | +40.33 | Rising  |
| health_dump_inbox       | 33.74      | 31.51     | -2.23  | Falling |

### Most Degraded Worker
The worker most at risk is the one with the **largest positive trend**, as it shows a consistent and significant increase in CPU usage. This is `unit_c_diagnostics` on **worker3**, with a +40.33 percentage point increase over the observation window.

Although `busy_box_telemetry` has a higher *current* CPU (83.91%), its trend (+1.59) is far less severe than `unit_c_diagnostics`'s sharp rise.

**Conclusion**: The de

## Teardown

In [8]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

Gustavo owns this fleet - leaving it alone.
